## Chargments des bibliotheques

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score

## Chargement des donnees

In [2]:
try: 
    print('Chargement en cours ...')
    train = pd.read_csv('data/train.csv')
    test = pd.read_csv('data/test.csv')
except:
    print('Erreur de chargment !')
else:
    print('Chargement Reussi !')

Chargement en cours ...
Chargement Reussi !


### Structure General du dataset

In [3]:
print(train.info())
print(train.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290081 entries, 0 to 1290080
Data columns (total 11 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   id                          1290081 non-null  object 
 1   period                      1290081 non-null  int64  
 2   operation                   1290081 non-null  object 
 3   amount                      1290081 non-null  float64
 4   origin_account              1290081 non-null  object 
 5   origin_balance_before       1290081 non-null  float64
 6   origin_balance_after        1290081 non-null  float64
 7   destination_account         1290081 non-null  object 
 8   destination_balance_before  1290081 non-null  float64
 9   destination_balance_after   1290081 non-null  float64
 10  fraud_flag                  1290081 non-null  int64  
dtypes: float64(5), int64(2), object(4)
memory usage: 108.3+ MB
None
id                            1290081
period       

### Copier la base d'entrainement pour securite

In [4]:
df_copy = train.copy()

## Traitement de donnee

#### Suppression des variables identite

In [5]:
df = df_copy.drop(columns=['id', 'origin_account', 'destination_account'])

### Separation de la base en train/test

In [6]:
## recuperer la variable cible
x = df.drop(columns=['fraud_flag'])
y = df['fraud_flag']

In [7]:
## decouper la base
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
## verification du decoupage
print(f"Taille Train {x_train.shape[0]} | Taille Test {x_test.shape[0]}")
print(f'Proportion de fraude (Train) {y_train.mean():.3f}')
print(f'Proportion de fraude (test) {y_test.mean():.3f}')

Taille Train 1032064 | Taille Test 258017
Proportion de fraude (Train) 0.100
Proportion de fraude (test) 0.100


### Encodage des variables qualitatives

In [8]:
## colonne categorielles
vars_quali = x_train.select_dtypes(include=['object']).columns
## noms variables quantitatives
vars_quanti = x_train.select_dtypes(exclude=['object', 'category']).columns
## instance oneHotEncoder
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
## encoder les variables train/test
x_train_cat = encoder.fit_transform(x_train[vars_quali])
x_test_cat = encoder.transform(x_test[vars_quali])
## creation de dataframe encoder
x_train_qual = pd.DataFrame(x_train_cat, columns=encoder.get_feature_names_out(vars_quali), index=x_train.index)
x_test_qual = pd.DataFrame(x_test_cat, columns=encoder.get_feature_names_out(vars_quali), index=x_test.index)
## concatener les bases
x_train_final = pd.concat([x_train[vars_quanti], x_train_qual], axis=1)
x_test_final = pd.concat([x_test[vars_quanti], x_test_qual], axis=1)

#### Regression Logistique (Modele Generalise)

In [ ]:
## ajout de la constante
x_train_sm = sm.add_constant(x_train_final)
x_test_sm = sm.add_constant(x_test_final)
## modele generalise
medele_general = sm.GLM(endog=y_train, exog=x_train_sm, family=sm.families.Binomial()).fit()
print(medele_general.summary())